# Explore NetCDF files from WRF-Chem modeled output

Authors: Vanessa Monteiro (ICTA-UAB), Qing Luo (ICTA-UAB), and Yi Wang (ICOS Carbon Portal)

Last updated: 2026-10-21

## 1. Introduction

<span style="font-size: 16px;"><b>What is NetCDF</b></span>

NetCDF (Network Common Data Form) is a widely used file format for storing and sharing multidimensional scientific data. Unlike traditional data formats such as CSV, NetCDF can efficiently organize data across multiple dimensions (e.g., timestamp, latitude, longitude, altitude). It also stores metadata that describe variables, units, coordinates, and other useful information needed to interpret the data.

<span style="font-size: 16px;"><b>About the dataset</b></span>

The dataset explored in this notebook contains simulated atmospheric carbon dioxide (CO<sub>2</sub>) concentrations generated by a high-resolution atmospheric transport and chemistry model ([WRF-Chem](https://www2.acom.ucar.edu/wrf-chem)). It provides hourly simulation results over the Barcelona city area for one month, across a three-dimensional spatial grid (latitude, longitude, and vertical). The dataset is openly available via [ICOS Cities Data Portal](https://citydata.icos-cp.eu/portal/).

<span style="font-size: 16px;"><b>What will we explore?</b></span>

This notebook provides a basic introduction to reading, opening, exploring NetCDF data and related metadata using Python. We will use ``icoscp_core`` library to retrieve and inspect the metadata associated with the data object from the ICOS Cities Data Portal. We will then open the dataset and examine its structure and contents using the ``xarray`` library. We will also use two interactive visualization tools to further explore the model simulation results.

## 2. Import modules
Here we import necessary python modules and scripts.

In [ ]:
from icoscp_core.icos import auth
from icoscp_core.cities import data as citydata, meta as citymeta
import requests
import xarray as xr
import pandas as pd
from scripts.nc_explorer import *

## 3. Find data objects and access metadata using ``icoscp_core`` library
In this section we retrieve all data objects from the Barcelona City model domain (for the WRF-Chem model) and access the related metadata. 

In [ ]:
# List station(s) in the Barcelona city domain
stations = citymeta.list_stations()
BCN = [s for s in stations if 'Barcelona_City' in s.uri]
pd.DataFrame(BCN)

### 3.1 Access data objects from a given station

In [ ]:
# List data objects from this station
dobj_list = citymeta.list_data_objects(station=BCN)
dobj_list

In [ ]:
# Here we only access the first data object of this station
dobj_uri = dobj_list[0].uri
dobj_uri

In [ ]:
# get all metadata from this data object
dobj_meta = citymeta.get_dobj_meta(dobj_uri)

### 3.2 Explore related metadata
Using the ``citymeta`` module, we can access metadata associated with a data object, such as its latest version, title, geospatial coverage, and recommended citation. This helps us identify and retrieve the most recent version of the data object and obtain accurate citation information for proper attribution.

Tip: to see what attribute you can get from the metadata, type `dobj_meta.` and then press tab.

In [ ]:
# get all metadata from this data object
dobj_meta = citymeta.get_dobj_meta(dobj_uri)

In [ ]:
# get lastest version of this data object (in case uri provided above is outdated)
dobj_meta.latestVersion

In [ ]:
# get title
dobj_meta.references.title

In [ ]:
# get citation string
dobj_meta.references.citationString

In [ ]:
# get BibTex
print(dobj_meta.references.citationBibTex)

In [ ]:
# get file name
dobj_meta.fileName

In [ ]:
# get geospatial coverage (four corners_
dobj_meta.coverageGeo

In [ ]:
# get temporal coverage
dobj_meta.specificInfo.temporal

In [ ]:
# get station network
dobj_meta.specificInfo.station.specificInfo.network

## 4. Access NetCDF files (with given uri) using OPeNDAP
Here we will open the NetCDF using OPeNDAP service, which allows users to access and explore the data remotely without downloading the entire file to their local computer.

### 4.1 Authentication
To access the NetCDF data, you need an ICOS Carbon Portal account. If you do not have one, you can register at [ICOS Carbon Portal Authentication Service](https://cpauth.icos-cp.eu/login/). Run the cell below to authenticate your account. Authentication is only required once, as your credentials will be saved locally in your Jupyter environment for future access.

In [ ]:
# Uncomment to authenticate
# Only need to execute this cell once for one account
# The authentication info will be saved to your Jupyter
# Comment out after successful authentication
# auth.init_config_file()

In [ ]:
# Find your token and validate if your token is valid
token = auth.get_token()
if token.is_fresh(): print('Token is valid')

### 4.2 Helper function

In [ ]:
# Define a helper function to open a netCDF dataset over OPeNDAP with authenticated session, without downloading the file first
session = requests.Session()
session.headers.update({
    'Authorization': 'Bearer '+auth.get_token().cookie_value,
})

# Helper function for url construction and pydap access
def xrDataSet(dobj_uri:str):
    objId = dobj_uri.split("/")[-1]
    url = 'dap2://opendap.icos-cp.eu/' + objId
    return xr.open_dataset(url, engine='pydap', backend_kwargs={'session': session})

### 4.3 Explore more about the netCDF dataset itself using ``xarray`` module
The ``xarray`` module provides a convenient way to explore and analyze multidimensional NetCDF datasets. In the following cells, we open the NetCDF dataset as an xarray dataset object (``xrDataSet``) and display its structure and contents. This allows us to inspect the dataset's dimensions, coordinates, data variables, attributes, and other information without downloading the entire dataset.

In [ ]:
# read in dataset as a xarray dataset
ds = xrDataSet(dobj_uri)

In [ ]:
# check out dataset
ds

## 5. Interactive panels
From the previous steps, we can see that this NetCDF dataset contains four dimensions: time, latitude, longitude, and vertical level (eta). To explore the key data variables in this four-dimensional dataset, we use two pre-defined interactive visualization tools to examine the spatial and temporal variations of CO<sub>2</sub> concentrations.

In the WRF-Chem simulations, a background CO<sub>2</sub> concentration of 400 ppm is used to represent the approximately uniform atmospheric CO<sub>2</sub> background. This background value is included in tracer 2 (CO<sub>2</sub> concentration from biosphere) and tracer 3 (CO<sub>2</sub> concentration from anthropogenic activities). Therefore, 400 ppm is subtracted from these two tracers when plotting the results to highlight the CO<sub>2</sub> concentration variations associated with biospheric and anthropogenic contributions, respectively. The subtraction is applied only for visualization and does not modify the original NetCDF data.

### 5.1 Explore the spatial distribution of CO<sub>2</sub> concentrations

The ``MapPanels`` function generates four map panels showing the spatial distribution of CO<sub>2</sub> concentrations. You can select different timestamps and vertical eta levels from the dropdown menus to explore the variations across the model domain.

Note: Vertial level (eta) is a dimensionless, terrain-following vertical coordinate used in the WRF-Chem model to represent atmospheric levels. It generally ranges from 1 near the surface to 0 at the model top. These eta levels are pressure-based and do not correspond to fixed altitudes, meaning that the same eta level may represent different heights at different locations.

In [ ]:
# choose timestamp and vertical level eta
MapPanels(ds).show()

### 5.2 Explore time series at a given location
The ``WRFExplorer`` function provides an interactive map for exploring the temporal variations of CO<sub>2</sub> concentrations at specific locations. You can select a vertical eta level and concentration variable from the dropdown menus, then click on the map to display the corresponding time series.

Note: The time series plot may take some time to update.

In [ ]:
# Choose timestamp, vertical height, and concentration from drop down menu
# Click on map to see time series
WRFExplorer(ds).show()

## Data source and acknowledgment

Luo, Q., Monteiro, V., Villalba Mendez, G. (2026). WRF-Chem simulated atmospheric CO2 concentrations for Barcelona, December 2023, 2023-12-01–2023-12-31, ICOS Cities, https://hdl.handle.net/11676/LJ7II3ZiD4_YP_uVCuUKUmqk

We  acknowledge the valuable input and feedback from Claudio D'Onofrio, Margareta Hellstrom, and Ute Karstens (ICOS Carbon Porta). OpenAI's ChatGPT (GPT-6) was used to assist with code development and text improvement. All AI-generated outputs were reviewed and validated by the authors.